# 03 — Controlled, label-blind retrieval corruption
For each dev claim, replace the highest-ranked retrieved document with a distractor drawn from a predeclared lexical-similarity band among lower-ranked documents. Selection uses retrieval scores and rank only; it never reads gold evidence IDs, labels or rationale sentences. The random seed is fixed.

In [ ]:
from pathlib import Path
import json, random, pandas as pd
BASE=Path("/content/scifact_study")
def read_jsonl(p):
    with open(p,encoding="utf-8") as f: return [json.loads(x) for x in f if x.strip()]
clean=read_jsonl(BASE/"results/clean_rankings.jsonl")
SEED=37
TOP_K=3
BAND_LOW, BAND_HIGH = 4, 20  # 1-based ranks in clean ranking
rng=random.Random(SEED)


In [ ]:
corrupted=[]
for r in clean:
    ids=r["doc_ids"]
    scores=r["scores"]
    if len(ids) < BAND_HIGH:
        raise ValueError(f"Need at least {BAND_HIGH} ranked candidates for claim {r['claim_id']}")
    # Distractor pool is defined by rank, without gold-label/evidence inspection.
    pool=list(range(BAND_LOW-1, BAND_HIGH))
    chosen=rng.choice(pool)
    corrupt_ids=ids[:TOP_K].copy()
    corrupt_scores=scores[:TOP_K].copy()
    corrupt_ids[0]=ids[chosen]
    corrupt_scores[0]=scores[chosen]
    corrupted.append({
        "claim_id":r["claim_id"],
        "doc_ids":corrupt_ids,
        "scores":corrupt_scores,
        "injected_doc_id":ids[chosen],
        "injected_original_rank":chosen+1,
        "reserve_doc_ids":ids[TOP_K:],
        "reserve_scores":scores[TOP_K:]
    })
print("Corrupted claims:",len(corrupted))
print("Example clean:",clean[0]["doc_ids"][:TOP_K])
print("Example corrupted:",corrupted[0]["doc_ids"])


In [ ]:
out=BASE/"results"
with open(out/"corrupted_rankings.jsonl","w",encoding="utf-8") as f:
    for r in corrupted: f.write(json.dumps(r)+"\n")
pd.DataFrame([{"claim_id":r["claim_id"],"injected_doc_id":r["injected_doc_id"],
               "injected_original_rank":r["injected_original_rank"]} for r in corrupted]).to_csv(out/"corruption_manifest.csv",index=False)
print("Saved corruption outputs and manifest.")


## Interpretation
This is a deliberately controlled stress test, not a claim that real systems make errors in exactly this way. The corruption policy must remain frozen before examining gold outcomes. The same corrupted list will be used in the no-checkpoint and checkpoint conditions.